# Kirkwood gaps

The overnight sweep in `data/sweep/` puts five runs side by side. Each holds
100 000 test particles in the Sun--Jupiter restricted problem, dumped 64 times
on the way, and each changes exactly one thing:

| directory | e_max | years | perturbers |
|---|---|---|---|
| `e005` | 0.05 | 1 Myr | Jupiter |
| `e010_long` | 0.10 | **3 Myr** | Jupiter |
| `e020` | 0.20 | 1 Myr | Jupiter |
| `e030` | 0.30 | 1 Myr | Jupiter |
| `e010sat` | 0.10 | 1 Myr | Jupiter + Saturn |

The measure throughout is the **depletion ratio**: how many particles sit
within $\pm 0.025$ AU of a resonance at the end, divided by how many sat there
at the start. Four of the five resonances turn out to be a null result, and the
interesting part is how tightly they can be bounded.

All five runs share a seed, so they start from identical particles.

In [ ]:
import sys

sys.path.insert(0, "python")

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from kirkwood_io import read_epoch, elements, resonance_axes, semimajor_axis

SWEEP = Path("data/sweep")
HALF_WIDTH_AU = 0.025

def epoch_at(run, step):
    """The epoch nearest `step`; the runs were dumped on different strides."""
    files = sorted((SWEEP / run).glob("epoch_*.bin"),
                   key=lambda p: int(p.stem.split("_")[1]))
    return read_epoch(min(files, key=lambda p: abs(
        int(p.stem.split("_")[1]) - step)))


first = epoch_at("e005", 0)
RES = resonance_axes(semimajor_axis(first.planet_r[0], first.planet_v[0], first.G))
A0, _ = elements(first)


def depletion(run, step, label):
    """Counts around one resonance before and after.

    The error is Poisson on the final count, not binomial: the window at the
    end holds particles that drifted in from outside as well as survivors, so
    it is not a fixed set being tracked.
    """
    a, _ = elements(epoch_at(run, step))
    inside = np.abs(a - RES[label]) < HALF_WIDTH_AU
    n1 = int(inside.sum())
    n0 = int((np.abs(A0 - RES[label]) < HALF_WIDTH_AU).sum())
    return n0, n1, n1 / n0, np.sqrt(n1) / n0

## One resonance moves; four do not

At 1 Myr with $e_{\max} = 0.10$, only the 2:1 is visible. The other four sit
within a percent of doing nothing, and the error bars say that is a bound
rather than a coincidence: a window holds about 3300 particles, so anything
deeper than roughly 3.5 % would show at $2\sigma$.

The last two rows change $e_{\max}$ instead, and the 2:1 responds strongly.

In [ ]:
ORDER = ["2:1"] + [k for k in RES if k != "2:1"]

print(f"{'run':>10} {'e_max':>6}   " + "  ".join(f"{k:>13}" for k in ORDER))
for run, e_max in [("e005", "0.05"), ("e010_long", "0.10"),
                   ("e020", "0.20"), ("e030", "0.30")]:
    row = [f"{depletion(run, 36_525_000, k)[2]:6.3f}"
           f"±{depletion(run, 36_525_000, k)[3]:.3f}" for k in ORDER]
    print(f"{run:>10} {e_max:>6}   " + "  ".join(f"{c:>13}" for c in row))

print()
print(f"每个窗口初态约 {int((np.abs(A0 - RES['2:1']) < HALF_WIDTH_AU).sum())} 个粒子，"
      f"所以 1 sigma 约 {np.sqrt(3282) / 3282:.1%}")


## Is the 3:1 merely slower?

This is the question the earlier 20 000-particle sweep could not answer. If the
3:1 were just slower than the 2:1, tripling the run time should start to show
it. `e010_long` runs three times as long, and nothing appears.

The 2:1, by contrast, reaches its full depth inside the first million years and
then stops moving -- so the timescale on which a resonance of this kind acts is
shorter than 1 Myr, which is exactly why the 3:1's silence at 3 Myr is
informative rather than a matter of patience.

In [ ]:
print(f"{'Myr':>5}  " + "  ".join(f"{k:>14}" for k in RES))
history = {k: [] for k in RES}
for myr in [1, 2, 3]:
    row = []
    for label in RES:
        _, _, ratio, err = depletion("e010_long", int(myr * 36_525_000), label)
        history[label].append(ratio)
        row.append(f"{ratio:6.3f}±{err:.3f}")
    print(f"{myr:>5}  " + "  ".join(f"{c:>14}" for c in row))

fig, ax = plt.subplots(figsize=(6, 3.4))
for label, values in history.items():
    ax.plot([1, 2, 3], values, marker="o", label=label)
ax.axhline(1.0, color="0.7", lw=0.8, zorder=0)
ax.set_xlabel("time (Myr)")
ax.set_ylabel("depletion ratio")
ax.set_xticks([1, 2, 3])
ax.legend(ncol=5, fontsize=8, frameon=False)
ax.set_title("Only the 2:1 moves, and it is done within 1 Myr", fontsize=10)
fig.tight_layout()
plt.show()

## Saturn

Same seed, same 100 000 particles, one extra perturber. The two ratios differ
by less than $1\sigma$, which is the same answer the earlier sweep gave at a
fifth of the statistics -- now with an error bar on it.

In [ ]:
for run in ["e010_long", "e010sat"]:
    perturbers = "Jupiter + Saturn" if run == "e010sat" else "Jupiter"
    _, _, ratio, err = depletion(run, 36_525_000, "2:1")
    print(f"{run:>9}  {perturbers:>16}  2:1 -> {ratio:.3f} ± {err:.3f}")

difference = (depletion("e010sat", 36_525_000, "2:1")[2]
              - depletion("e010_long", 36_525_000, "2:1")[2])
combined = np.hypot(depletion("e010sat", 36_525_000, "2:1")[3],
                    depletion("e010_long", 36_525_000, "2:1")[3])
print(f"\ndifference {difference:+.3f} ± {combined:.3f}"
      f"  ->  {abs(difference) / combined:.1f} sigma")